In [21]:
import json

sample = json.load(open("/data/shanghong/oumi/grpo/data/sample.json"))
new_data = [sample]

with open("/data/shanghong/oumi/grpo/data/sample.jsonl", "w") as f:
    for d in new_data:
        f.write(json.dumps(d) + "\n")

In [11]:
import pandas as pd
from typing_extensions import override

from oumi.core.datasets.base_grpo_dataset import BaseExperimentalGrpoDataset
from oumi.core.types.conversation import Conversation


class CountdownDataset(BaseExperimentalGrpoDataset):
    """Dataset class for the `grpo_data_train` and `grpo_data_test` dataset.

    A sample from the dataset:
    {
        "messages": [
            {
                "content": "You are a helpful assistant. You first think about the reasoning process in the mind and then provide the user with the answer.",
                "role": "system"
            },
            {
                "content": "Using the numbers [73, 30, 38, 97], create an equation that equals 92. You can use basic arithmetic operations (+, -, *, /) and each number can only be used once. Show your work in <think> </think> tags. And return the final equation and answer in <answer> </answer> tags, for example <answer> (1 + 2) / 3 = 1 </answer>.",
                "role": "user"
            },
            {
                "content": "Let's think step by step.\n\nWe are given the numbers: [73, 30, 38, 97]  \nWe need to use each number exactly once with basic arithmetic operations (+, \u2212, \u00d7, \u00f7) to get a result of 92.\n\nOur goal: form an equation using all four numbers once, such that the result is 92.\n\nStep 1: Try to find combinations that get close to 92.\n\nLet\u2019s look at differences between numbers:\n\n- 97 \u2212 73 = 24  \n- 97 \u2212 38 = 59  \n- 97 \u2212 30 = 67  \n- 73 \u2212 38 = 35  \n- 73 \u2212 30 = 43  \n- 38 \u2212 30 = 8  \n\nNot immediately helpful.\n\nWhat if we try to build 92 via addition or subtraction?\n\nTry:  \n97 \u2212 30 = 67 \u2192 67 + 38 = 105 \u2192 too big  \n97 \u2212 38 = 59 \u2192 59 + 30 = 89 \u2192 89 + 73 = 162 \u2192 too big  \n73 + 30 = 103 \u2192 103 \u2212 38 = 65 \u2192 65 + 97 = 162\n\nTry: 97 \u2212 (73 \u2212 38) = 97 \u2212 35 = 62 \u2192 62 + 30 = 92! \u2705\n\nWait \u2014 let's verify:\n\n73 \u2212 38 = 35  \n97 \u2212 35 = 62  \n62 + 30 = 92\n\nYes! That works.\n\nSo the full expression is:\n\n97 \u2212 (73 \u2212 38) + 30 = 92\n\nNow, check if we used all numbers:  \n73 \u2705, 38 \u2705, 97 \u2705, 30 \u2705  \nEach used once.  \nOnly basic operations: subtraction and addition.\n\nSo this is valid.\n\nNow format as requested:\n\n<answer> 97 - (73 - 38) + 30 = 92 </answer>",
                "role": "assistant"
            }
        ],
        "metadata": {
            "target": 92,
            "nums": [
                73,
                30,
                38,
                97
            ]
        }
    }
    """  # noqa: E501

    default_dataset = "countdown_dataset"

    @override
    def transform(self, sample: pd.Series) -> dict:
        """Validate and transform the sample into Python `dict`."""
        sample_dict = sample.to_dict()
        output_dict = {
            "prompt": sample_dict["messages"],
            "nums": sample_dict["metadata"]["nums"],
            "target": sample_dict["metadata"]["target"],
        }
        return output_dict

    @override
    def transform_conversation(self, sample: pd.Series) -> Conversation:
        """Converts the input sample to a Conversation.

        Args:
            sample (dict): The input example.

        Returns:
            Conversation: The resulting conversation.

        """
        # Example is already in conversation format and only needs light processing.
        sample_dict = sample.to_dict()
        return Conversation.from_dict(sample_dict)

In [24]:
sample = pd.Series(sample)
print(
    CountdownDataset(
        dataset_path="/data/shanghong/oumi/grpo/data/sample.jsonl"
    ).transform(sample)
)
print("====")
print(
    CountdownDataset(
        dataset_path="/data/shanghong/oumi/grpo/data/sample.jsonl"
    ).transform_conversation(sample)
)

[2026-01-21 00:44:31,801][oumi][rank0][pid:98316][MainThread][INFO]][base_map_dataset.py:91] Creating map dataset (type: CountdownDataset)... dataset_name: 'countdown_dataset' dataset_path: '/data/shanghong/oumi/grpo/data/sample.jsonl'
[2026-01-21 00:44:32,054][oumi][rank0][pid:98316][MainThread][INFO]][base_map_dataset.py:426] Loaded DataFrame with shape: (1, 2). Columns:
messages    object
metadata    object
dtype: object
{'prompt': [{'content': 'You are a helpful assistant. You first think about the reasoning process in the mind and then provide the user with the answer.', 'role': 'system'}, {'content': 'Using the numbers [73, 30, 38, 97], create an equation that equals 92. You can use basic arithmetic operations (+, -, *, /) and each number can only be used once. Show your work in <think> </think> tags. And return the final equation and answer in <answer> </answer> tags, for example <answer> (1 + 2) / 3 = 1 </answer>.', 'role': 'user'}, {'content': "Let's think step by step.\n\nWe 

/opt/conda/envs/oumi/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu128 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
2026-01-21 02:05:49,629	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [6]:
import importlib
import sys

from oumi.core.registry import REGISTRY, RegistryType

sys.path.append("/data/shanghong/oumi")
importlib.import_module("grpo.countdown_dataset")  # trigger decorator

dataset_cls = REGISTRY.get_dataset("countdown_data")
print("registered:", dataset_cls)
print(
    "all datasets:",
    [
        key.name
        for key in REGISTRY._registry
        if key.registry_type == RegistryType.DATASET
    ],
)

registered: <class 'grpo.countdown_dataset.CountdownDataset'>
all datasets: ['debug_classfication', 'debug_pretraining', 'debug_sft', 'debug_dpo', 'debug_kto', 'tatsu-lab/alpaca_eval', 'oumi-ai/berrybench-v0.1.1', 'd1shs0ap/countdown', 'openai/gsm8k', 'oumi-ai/oumi-letter-count', 'oumi-ai/oumi-letter-count-clean', 'rar-medicine', 'rar-science', 'oumi-rlvr-rubric', 'trl-lib/tldr', 'trl-lib/kto-mix-14k', 'mlabonne/orpo-dpo-mix-40k', 'allenai/c4', 'allenai/dolma', 'tiiuae/falcon-refinedweb', 'huggingfacefw/fineweb-edu', 'eleutherai/pile', 'togethercomputer/redpajama-data-1t', 'togethercomputer/redpajama-data-v2', 'cerebras/slimpajama-627b', 'bigcode/starcoderdata', 'bigcode/the-stack', 'roneneldan/tinystories', 'nampdn-ai/tiny-textbooks', 'wikimedia/wikipedia', 'salesforce/wikitext', 'pleias/youtube-commons', 'tatsu-lab/alpaca', 'yahma/alpaca-cleaned', 'cohereforai/aya_dataset', 'nvidia/chatqa-training-data', 'nvidia/chatqa-training-data/tatqa-others', 'nvidia/chatqa-training-data/tatqa-a